# Tarea 1 – Parte 1: Recolección y limpieza (La República)

**Pregunta guía:** ¿Se pueden convertir titulares peruanos en un clasificador de noticias?

**Estudiante:** Karen Macedo

| Campo | Valor |
| :-- | :-- |
| Fecha de descarga | se imprime en la celda de configuración (`FECHA_DESCARGA`) |
| Opción elegida | **A – La República** |
| URL base | `https://larepublica.pe` |
| RSS | `https://larepublica.pe/rss/{seccion}.xml` |
| Sitemaps | `https://larepublica.pe/sitemap/{seccion}.xml` |
| Completar histórico | índice CDX de Internet Archive (`web.archive.org/cdx`), solo para *listar* URLs de La República |

**Categorías fijas:** `politica`, `economia`, `deportes`, `ciencia_tecnologia`. Lo que no entra en ninguna se descarta (no se crea una quinta categoría).

**Ética del scraping:** reviso `robots.txt`, uso un User-Agent descriptivo, espero 1–2 s entre pedidos y reintento como máximo 1 vez. Los RSS y sitemaps solo traen las últimas notas, por eso completo con URLs antiguas listadas en Internet Archive, pero cada nota la descargo de `larepublica.pe` (página individual, que sí está permitida).

In [ ]:
import json, random, re, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
from bs4 import BeautifulSoup

# png para que los gráficos se vean en GitHub (necesita kaleido==0.2.1)
pio.renderers.default = "png+plotly_mimetype"
pio.templates.default = "plotly_white"

BASE = "https://larepublica.pe"
HEADERS = {"User-Agent": "TareaDataSciencePython/1.0 (proyecto academico de estudiante; scraping educativo)"}
FECHA_DESCARGA = datetime.now().strftime("%Y-%m-%d")
META_POR_CATEGORIA = 100      # 4 x 100 = 400 noticias
SEED = 42
random.seed(SEED)

RAW = Path("datos/raw")
RAW.mkdir(parents=True, exist_ok=True)

ultimo = {"t": 0.0}

def get(url, pausa=(1, 2), timeout=40):
    """GET con pausa de 1-2 s entre pedidos y máximo 1 reintento."""
    for intento in range(2):
        espera = random.uniform(*pausa) - (time.time() - ultimo["t"])
        if espera > 0:
            time.sleep(espera)
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            ultimo["t"] = time.time()
            if r.status_code >= 500 and intento == 0:
                continue
            return r
        except requests.RequestException as e:
            ultimo["t"] = time.time()
            if intento == 1:
                print("Error:", url, e)
    return None

print("Fecha de descarga:", FECHA_DESCARGA)

## 1. Revisión de `robots.txt`

In [ ]:
r = get(f"{BASE}/robots.txt")
(RAW / "robots_larepublica.txt").write_text(r.text, encoding="utf-8")
robots = RobotFileParser()
robots.parse(r.text.splitlines())

for linea in r.text.splitlines():
    if any(k in linea for k in ("/rss", "/sitemap", "/pag/", "/tag/", "/buscador")):
        print("  ", linea)

pruebas = [f"{BASE}/rss/politica.xml", f"{BASE}/sitemap/deportes.xml",
           f"{BASE}/politica/2026/10/01/ejemplo-de-nota-hnews-1"]
for u in pruebas:
    print("PERMITIDO" if robots.can_fetch(HEADERS["User-Agent"], u) else "BLOQUEADO", u)

Los RSS, sitemaps y páginas de cada nota están permitidos. No uso la paginación ni el buscador del sitio.

## 2. Extracción – fuente 1: RSS y sitemaps oficiales

In [ ]:
SECCIONES = ["politica", "economia", "deportes", "ciencia", "tecnologia", "mundo", "sociedad"]
PATRON_NOTA = re.compile(r"^https?://larepublica\.pe(?::\d+)?/(?:amp/)?([a-z0-9-]+)/(\d{4})/(\d{2})/(\d{2})/([^/?#]+)/?$")

def normalizar(url):
    m = PATRON_NOTA.match(url.strip())
    if not m:
        return None
    sec, y, mes, d, slug = m.groups()
    return f"{BASE}/{sec}/{y}/{mes}/{d}/{slug}"

notas = {}   # url -> {"titular", "bajada", "fuente"}

for sec in SECCIONES:
    r = get(f"{BASE}/rss/{sec}.xml")
    if r is None or r.status_code != 200:
        print(f"RSS {sec}: no disponible")
        continue
    (RAW / f"rss_{sec}.xml").write_text(r.text, encoding="utf-8")
    sopa = BeautifulSoup(r.text, "xml")
    n = 0
    for item in sopa.find_all("item"):
        url = normalizar(item.link.get_text(strip=True)) if item.link else None
        if url:
            notas.setdefault(url, {"titular": item.title.get_text(strip=True) if item.title else "",
                                   "bajada": item.description.get_text(strip=True) if item.description else "",
                                   "fuente": "rss"})
            n += 1
    print(f"RSS {sec:<11} {n:>3} notas")

for sec in SECCIONES:
    r = get(f"{BASE}/sitemap/{sec}.xml")
    if r is None or r.status_code != 200:
        print(f"Sitemap {sec}: no disponible")
        continue
    (RAW / f"sitemap_{sec}.xml").write_text(r.text, encoding="utf-8")
    n = 0
    for loc in BeautifulSoup(r.text, "xml").find_all("loc"):
        url = normalizar(loc.get_text(strip=True))
        if url and url not in notas:
            notas[url] = {"titular": "", "bajada": "", "fuente": "sitemap"}
            n += 1
    print(f"Sitemap {sec:<7} {n:>3} notas nuevas")

print("Total fuente 1:", len(notas))

### Tabla de equivalencias `seccion_original` → `etiqueta`
La sección se toma del primer segmento de la URL (ej. `larepublica.pe/deportes/...`).

In [ ]:
MAPEO = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deportes",
    "ciencia": "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
display(pd.DataFrame(MAPEO.items(), columns=["seccion_original", "etiqueta"]))
print("Cualquier otra sección (mundo, sociedad, espectáculos, etc.) se descarta.")

def seccion_de(url):
    return url.split("/")[3]

## 3. Extracción – fuente 2: completar con URLs históricas
Con RSS y sitemaps no llego a 100 por categoría, así que listo URLs antiguas de las mismas secciones con el CDX de Internet Archive.

In [ ]:
CDX = "http://web.archive.org/cdx/search/cdx"
candidatas = {}
for sec in MAPEO:
    for anio in [2026, 2025]:
        archivo = RAW / f"cdx_{sec}_{anio}.txt"
        if archivo.exists():
            texto = archivo.read_text(encoding="utf-8")
        else:
            r = get(f"{CDX}?url=larepublica.pe/{sec}/{anio}/*&fl=original&collapse=urlkey"
                    f"&filter=statuscode:200&limit=1500", pausa=(2, 3), timeout=150)
            texto = r.text if (r is not None and r.status_code == 200) else ""
            if texto:
                archivo.write_text(texto, encoding="utf-8")
        for linea in texto.splitlines():
            u = normalizar(linea)
            if u and seccion_de(u) == sec and u not in notas:
                candidatas[u] = sec
    print(f"CDX {sec:<11} acumuladas: {len(candidatas)}")

### Descarga de cada nota
Saco titular (`og:title`) y bajada (`og:description`) de la página. Todo se guarda en `datos/raw/articulos_raw.jsonl` para no repetir pedidos si se vuelve a ejecutar.

In [ ]:
RAW_JSONL = RAW / "articulos_raw.jsonl"
cache = {}
if RAW_JSONL.exists():
    for l in RAW_JSONL.read_text(encoding="utf-8").splitlines():
        d = json.loads(l)
        cache[d["url"]] = d

def limpiar(t):
    return re.sub(r"\s+", " ", t or "").strip()

def descargar(url):
    if url in cache:
        return cache[url]
    d = {"url": url, "titular": "", "bajada": "", "error": None}
    if not robots.can_fetch(HEADERS["User-Agent"], url):
        d["error"] = "bloqueado por robots.txt"
    else:
        r = get(url)
        if r is None:
            d["error"] = "sin respuesta"
        elif r.status_code != 200:
            d["error"] = f"HTTP {r.status_code}"
        else:
            s = BeautifulSoup(r.text, "lxml")
            meta = lambda p: (s.find("meta", attrs={"property": p}) or {}).get("content", "")
            d["titular"] = limpiar(meta("og:title") or (s.h1.get_text() if s.h1 else ""))
            d["bajada"] = limpiar(meta("og:description"))
    cache[url] = d
    with RAW_JSONL.open("a", encoding="utf-8") as f:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
    return d

def validas():
    cuenta = {c: 0 for c in CATEGORIAS}
    for u, n in notas.items():
        cat = MAPEO.get(seccion_de(u))
        if cat and n["titular"] and n["bajada"] and not n.get("error"):
            cuenta[cat] += 1
    return cuenta

# (a) notas del sitemap que no traen titular
for u, n in list(notas.items()):
    if n["fuente"] == "sitemap" and MAPEO.get(seccion_de(u)):
        d = descargar(u)
        n.update(titular=d["titular"], bajada=d["bajada"], error=d["error"])
print("Después de fuente 1:", validas())

# (b) completar cada categoría con las históricas (sorteo con semilla 42)
por_cat = {c: [] for c in CATEGORIAS}
for u, sec in sorted(candidatas.items()):
    por_cat[MAPEO[sec]].append(u)
rng = random.Random(SEED)
for cat, urls in por_cat.items():
    rng.shuffle(urls)
    for u in urls:
        if validas()[cat] >= META_POR_CATEGORIA:
            break
        d = descargar(u)
        notas[u] = {"titular": d["titular"], "bajada": d["bajada"], "fuente": "historico", "error": d["error"]}
    print(cat, "->", validas()[cat])

## 4. Limpieza

In [ ]:
bruto = pd.DataFrame([{
    "titular": limpiar(n["titular"]), "bajada": limpiar(n["bajada"]), "medio": "La República",
    "seccion_original": seccion_de(u), "url": u, "fecha": "-".join(u.split("/")[4:7]),
    "etiqueta": MAPEO.get(seccion_de(u)), "error": n.get("error"),
} for u, n in notas.items()])
print("Registros brutos:", len(bruto))

descartes = []
d = bruto.copy()
pasos = [
    ("Sección fuera de las 4 categorías", lambda x: x["etiqueta"].isna()),
    ("Nota inaccesible (HTTP ≠ 200 / sin respuesta)", lambda x: x["error"].notna()),
    ("Titular vacío", lambda x: x["titular"] == ""),
    ("Sin bajada", lambda x: x["bajada"] == ""),
    ("URL duplicada", lambda x: x.duplicated("url")),
    ("Titular duplicado", lambda x: x.duplicated("titular")),
]
for motivo, filtro in pasos:
    malos = filtro(d)
    descartes.append((motivo, int(malos.sum())))
    d = d[~malos]

exceso = d.groupby("etiqueta").cumcount() >= META_POR_CATEGORIA
descartes.append((f"Exceso sobre {META_POR_CATEGORIA} por categoría", int(exceso.sum())))
d = d[~exceso]

reporte = pd.DataFrame(descartes, columns=["motivo", "registros_descartados"])
display(reporte)
print("Secciones descartadas:", bruto.loc[bruto["etiqueta"].isna(), "seccion_original"].value_counts().to_dict())

COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
df = d[COLUMNAS].reset_index(drop=True)
print("Noticias finales:", len(df))
print(df["etiqueta"].value_counts())

## 5. Análisis exploratorio (Plotly Express)

In [ ]:
conteo = df["etiqueta"].value_counts().reindex(CATEGORIAS).reset_index()
conteo.columns = ["etiqueta", "n"]
titulo = (f"Las 4 categorías quedan balanceadas ({conteo.n.min()}–{conteo.n.max()} noticias c/u): "
          f"el modelo no tendrá una clase dominante")
fig = px.bar(conteo, x="etiqueta", y="n", text="n", color="etiqueta", title=titulo,
             labels={"etiqueta": "Categoría", "n": "Noticias"})
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
df["mes"] = pd.to_datetime(df["fecha"]).dt.to_period("M").astype(str)
por_mes = df.groupby(["mes", "etiqueta"]).size().reset_index(name="n")
mes_top = df["mes"].value_counts().idxmax()
titulo = (f"Las noticias van de {df['fecha'].min()} a {df['fecha'].max()}; "
          f"el mes con más notas es {mes_top} (efecto de los RSS recientes)")
fig = px.bar(por_mes, x="mes", y="n", color="etiqueta", title=titulo,
             category_orders={"etiqueta": CATEGORIAS},
             labels={"mes": "Mes de publicación", "n": "Noticias", "etiqueta": "Categoría"})
fig.update_xaxes(type="category")
fig.show()

In [ ]:
df["largo_titular"] = df["titular"].str.len()
mediana = df.groupby("etiqueta")["largo_titular"].median()
titulo = (f"La mayoría de titulares mide ~{df.largo_titular.median():.0f} caracteres; "
          f"{mediana.idxmax()} tiene los más largos y {mediana.idxmin()} los más cortos")
fig = px.histogram(df, x="largo_titular", color="etiqueta", nbins=40, barmode="overlay", opacity=0.6,
                   title=titulo, category_orders={"etiqueta": CATEGORIAS},
                   labels={"largo_titular": "Caracteres del titular", "etiqueta": "Categoría"})
fig.show()

## 6. Validación: 5 titulares más largos y 5 más cortos

In [ ]:
pd.set_option("display.max_colwidth", 200)
print("Rango del histograma:", df.largo_titular.min(), "a", df.largo_titular.max(), "caracteres")
display(df.nlargest(5, "largo_titular")[["largo_titular", "etiqueta", "titular"]])
display(df.nsmallest(5, "largo_titular")[["largo_titular", "etiqueta", "titular"]])

Los extremos coinciden con los bordes del histograma. Los más largos son titulares explicativos (con citas o varias ideas) y los más cortos suelen ser de deportes o notas breves. Son titulares reales, no errores de extracción.

## 7. Exportación

In [ ]:
df[COLUMNAS].to_csv("datos/noticias.csv", index=False, encoding="utf-8")
final = pd.read_csv("datos/noticias.csv")
assert list(final.columns) == COLUMNAS and len(final) >= 350
print("df.shape =", final.shape)
final.head()